# Stg Layer — restaurant_place_details (Google Places Place Details, refreshed monthly)

For every restaurant that `stg_restaurant_geocode` matched to a Google place, read the **Place Details (New)** fields that the dashboard needs and keep the latest answer in `${silver}.stg_restaurant_place_details` (one row per `place_id`).

**Fields:** `rating`, `userRatingCount` (Google stars and number of reviews), `priceRange` (e.g. 200,000 → 800,000 ₫ per person), `priceLevel`, `primaryType` / `primaryTypeDisplayName` (e.g. `chinese_restaurant` / "Nhà hàng Quảng Đông", used for the cuisine), `types`, `googleMapsUri`, and the first photo's resource name (`photos[0].name`, for the dashboard later).

**Flow:**
1. Matched `place_id`s from the geocode cache that were **never fetched** or were fetched more than `REFRESH_DAYS` ago (ratings change, so each place is re-read about once a month); never-fetched first
2. Call `GET places/{place_id}` within the **daily** and **monthly** caps (counted over every run in `stg_restaurant_place_details_calls`); the rest wait for the next run
3. MERGE into the table (update on `place_id`). HTTP errors are **not** stored, so they are retried next run; 404 is stored as `fetch_status = not_found`.

**Quota:** on HTTP 429 the run stops calling the API but does not fail. A rejected key (401 / 403) fails the task.

**Cost:** rating / userRatingCount / priceRange / priceLevel are **Enterprise** fields, so each call is billed as **Place Details Enterprise** (1,000 free calls per month at the time of writing; check the Google Maps pricing page). `MONTHLY_CALL_LIMIT = 900` keeps the bill at $0; ~140 places × 1 refresh per month uses ~15% of it. Text Search in `stg_restaurant_geocode` stays Pro.

**Secret:** `maps/api_key` (Places API (New)).

In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

GEOCODE_TABLE = f"{SILVER}.stg_restaurant_geocode"
DETAILS_TABLE = f"{SILVER}.stg_restaurant_place_details"
CALLS_TABLE = f"{SILVER}.stg_restaurant_place_details_calls"  # API calls per run, for the caps
DAILY_CALL_LIMIT = 60       # Place Details calls per Vietnam day, over all runs
MONTHLY_CALL_LIMIT = 900    # per Vietnam calendar month: stays under the 1,000 free Enterprise calls
REFRESH_DAYS = 30           # re-read a place after this many days
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
# Enterprise fields (rating, userRatingCount, priceLevel, priceRange) make every call Place Details Enterprise
FIELD_MASK = ("id,displayName,primaryType,primaryTypeDisplayName,types,rating,userRatingCount,"
              "priceLevel,priceRange,googleMapsUri,photos")

print(f"Geocode source: {GEOCODE_TABLE}")
print(f"Delta target: {DETAILS_TABLE} · max {DAILY_CALL_LIMIT} calls per day, {MONTHLY_CALL_LIMIT} per month")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DETAILS_TABLE} (
        place_id                   STRING,
        fetch_status               STRING,
        place_name                 STRING,
        primary_type               STRING,
        primary_type_display_name  STRING,
        place_types                ARRAY<STRING>,
        rating                     DOUBLE,
        rating_count               INT,
        price_level                STRING,
        price_start_vnd            BIGINT,
        price_end_vnd              BIGINT,
        google_maps_uri            STRING,
        photo_name                 STRING,
        fetched_at                 TIMESTAMP
    )
    USING DELTA
    COMMENT 'Google Place Details per place_id (latest read, refreshed every REFRESH_DAYS). fetch_status = ok / not_found.'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CALLS_TABLE} (
        run_ts    TIMESTAMP,
        call_date DATE,
        api_calls INT,
        stored    INT,
        errors    INT
    )
    USING DELTA
    COMMENT 'Place Details calls per run; sums per day / month enforce DAILY_CALL_LIMIT / MONTHLY_CALL_LIMIT.'
""")

# --- Places due: never fetched first, then the oldest reads ---
todo = spark.sql(f"""
    SELECT g.place_id, d.fetched_at
    FROM (SELECT DISTINCT place_id FROM {GEOCODE_TABLE} WHERE match_status = 'matched' AND place_id IS NOT NULL) g
    LEFT JOIN {DETAILS_TABLE} d ON d.place_id = g.place_id
    WHERE d.fetched_at IS NULL OR d.fetched_at < current_timestamp() - INTERVAL {REFRESH_DAYS} DAYS
    ORDER BY d.fetched_at NULLS FIRST, g.place_id
""").collect()

# --- Caps: calls already made today / this month (Vietnam time) by earlier runs ---
today = spark.sql(f"SELECT to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}')) AS d").collect()[0]["d"]
used = spark.sql(f"""
    SELECT COALESCE(SUM(CASE WHEN call_date = DATE'{today}' THEN api_calls END), 0) AS day,
           COALESCE(SUM(api_calls), 0) AS month
    FROM {CALLS_TABLE}
    WHERE call_date >= trunc(DATE'{today}', 'MM')
""").collect()[0]
budget = max(0, min(DAILY_CALL_LIMIT - used["day"], MONTHLY_CALL_LIMIT - used["month"]))
print(f"Places due: {len(todo)} · calls today: {used['day']}/{DAILY_CALL_LIMIT} · this month: {used['month']}/{MONTHLY_CALL_LIMIT} · this run may call: {budget}")

In [ ]:
# --- Call Place Details (New) ---
import requests

API_KEY = dbutils.secrets.get("maps", "api_key")


def money_vnd(m):
    """Money {currencyCode, units, nanos} -> whole VND (None if missing or another currency)."""
    if not m or m.get("currencyCode", "VND") != "VND":
        return None
    return int(m.get("units", 0))


def place_details(place_id):
    resp = requests.get(
        f"https://places.googleapis.com/v1/places/{place_id}",
        params={"languageCode": "vi", "regionCode": "VN"},
        headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELD_MASK},
        timeout=30,
    )
    if resp.status_code == 404:
        return {"place_id": place_id, "fetch_status": "not_found"}, None
    if resp.status_code != 200:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
    p = resp.json()
    price = p.get("priceRange") or {}
    photos = p.get("photos") or []
    return {
        "place_id": place_id, "fetch_status": "ok",
        "place_name": (p.get("displayName") or {}).get("text"),
        "primary_type": p.get("primaryType"),
        "primary_type_display_name": (p.get("primaryTypeDisplayName") or {}).get("text"),
        "place_types": sorted(p.get("types", [])),
        "rating": float(p["rating"]) if p.get("rating") is not None else None,  # 4 comes back as an int
        "rating_count": p.get("userRatingCount"),
        "price_level": p.get("priceLevel"),
        "price_start_vnd": money_vnd(price.get("startPrice")),
        "price_end_vnd": money_vnd(price.get("endPrice")),
        "google_maps_uri": p.get("googleMapsUri"),
        "photo_name": photos[0].get("name") if photos else None,
    }, None


results, errors, api_calls = [], [], 0
for row in todo[:budget]:
    result, error = place_details(row["place_id"])
    api_calls += 1
    if error and error.startswith(("HTTP 401", "HTTP 403")):
        raise RuntimeError(f"Places API rejected the key (check secret maps/api_key and the API restrictions): {error}")
    if error and error.startswith("HTTP 429"):
        errors.append(f"{row['place_id']}: {error}")
        print(f"⏸️ Places API quota reached after {api_calls} call(s); the rest waits for the next run")
        break
    if error:
        errors.append(f"{row['place_id']}: {error}")
    else:
        results.append(result)
print(f"API calls: {api_calls} · stored: {len(results)} · errors (retried next run): {len(errors)}")
for e in errors[:5]:
    print("  ⚠️", e[:300])
if len(todo) > budget:
    print(f"⏸️ Cap: {len(todo) - budget} place(s) wait for a later run")

# Record the calls first, so the caps hold even if a later step fails
spark.sql(f"""
    INSERT INTO {CALLS_TABLE}
    VALUES (current_timestamp(), DATE'{today}', {api_calls}, {len(results)}, {len(errors)})
""")

In [ ]:
# --- MERGE into the table (latest read wins) ---
from pyspark.sql.functions import current_timestamp

schema = spark.table(DETAILS_TABLE).schema
if results:
    cols = [f for f in schema if f.name != "fetched_at"]
    rows = [{f.name: r.get(f.name) for f in cols} for r in results]
    new_df = spark.createDataFrame(rows, schema=", ".join(f"{f.name} {f.dataType.simpleString()}" for f in cols))
    new_df.withColumn("fetched_at", current_timestamp()).createOrReplaceTempView("details_src")
    spark.sql(f"""
        MERGE INTO {DETAILS_TABLE} AS t
        USING details_src AS s ON t.place_id = s.place_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
print(f"✅ Merged {len(results)} reads into {DETAILS_TABLE}")

In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id) AS ids,
           COUNT_IF(rating IS NOT NULL) AS with_rating,
           COUNT_IF(price_start_vnd IS NOT NULL) AS with_price_range,
           COUNT_IF(price_level IS NOT NULL) AS with_price_level,
           COUNT_IF(photo_name IS NOT NULL) AS with_photo
    FROM {DETAILS_TABLE}
""").collect()[0]
print(f"📊 {DETAILS_TABLE}: {stats['n']} places · rating {stats['with_rating']} · price range {stats['with_price_range']} "
      f"· price level {stats['with_price_level']} · photo {stats['with_photo']}")
print("📊 Grain check (place_id): " + ("✅" if stats["n"] == stats["ids"] else "❌ duplicates"))
display(spark.sql(f"""
    SELECT primary_type, primary_type_display_name, COUNT(*) AS n
    FROM {DETAILS_TABLE} GROUP BY 1, 2 ORDER BY n DESC
"""))
dbutils.jobs.taskValues.set("api_calls", api_calls)
print("\n--- stg_restaurant_place_details complete ---")